# 03_grid_events_autoloader

## Purpose

Load raw grid event CSV files from the governed landing volume into the Bronze Delta table using Auto Loader.

This notebook preserves event identity, timestamps, severity-related fields, asset references, and source metadata for later Silver and Gold work.

In [0]:
from pyspark.sql import functions as F

In [0]:
# Project configuration

catalog = "vattenfall_dev"
schema = "raw"

landing_volume = "landing"
checkpoint_volume = "checkpoints"

source_domain = "grid_events"

landing_subpath = "grid_events"
checkpoint_subpath = "grid_events_checkpoint"
schema_tracking_subpath = "grid_events_schema"

bronze_table_name = "bronze_grid_events"
bronze_table = f"{catalog}.{schema}.{bronze_table_name}"

In [0]:
# Build governed Unity Catalog volume paths

landing_path = f"/Volumes/{catalog}/{schema}/{landing_volume}/{landing_subpath}"
checkpoint_path = f"/Volumes/{catalog}/{schema}/{checkpoint_volume}/{checkpoint_subpath}"
schema_path = f"/Volumes/{catalog}/{schema}/{checkpoint_volume}/{schema_tracking_subpath}"

print("Source domain:", source_domain)
print("Landing path:", landing_path)
print("Checkpoint path:", checkpoint_path)
print("Schema path:", schema_path)
print("Bronze target table:", bronze_table)

In [0]:
# Inspect grid event landing files before ingestion

display(dbutils.fs.ls(landing_path))

In [0]:
# Read grid event files with Auto Loader

grid_events_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(landing_path)
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

In [0]:
# Reset Day 1 prototype table before creating the Day 2 Auto Loader Bronze table

spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")

print("Dropped existing Bronze table if present:", bronze_table)

In [0]:
# Write grid event records to Bronze Delta table

query = (
    grid_events_stream_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("Bronze ingestion completed for:", source_domain)
print("Bronze target table:", bronze_table)

In [0]:
# Validate Bronze grid events table

bronze_df = spark.table(bronze_table)

print("Rows in bronze after ingestion:", bronze_df.count())
print("Columns in bronze:", bronze_df.columns)

display(bronze_df.limit(20))

In [0]:
# Check ingestion metadata columns

required_metadata_columns = ["ingestion_ts", "source_file"]

for column_name in required_metadata_columns:
    if column_name in bronze_df.columns:
        print(f"Metadata column present: {column_name}")
    else:
        raise ValueError(f"Missing metadata column: {column_name}")